# Estimating $\pi$ with Monte Carlo — darts in a circle

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

A tiny, self-contained illustration of **Monte Carlo integration**: we estimate the number
$\pi$ by throwing random "darts" into a square and counting how many land inside the inscribed
circle. No molecules this time — just the essence of the Monte Carlo idea, and a look at how
**slowly** a random estimate converges.

## Theory in brief

### The dartboard method
Take a square of side $2r$ and the circle of radius $r$ inscribed in it. Their areas are in the
ratio

$$\frac{\text{area of circle}}{\text{area of square}} = \frac{\pi r^2}{(2r)^2} = \frac{\pi}{4}.$$

If we scatter points **uniformly at random** over the square, the *fraction* that fall inside the
circle approaches this ratio. So

$$\pi \approx 4\,\frac{N_{\text{inside}}}{N_{\text{total}}}.$$

This is Monte Carlo integration in miniature: a geometric quantity (an area, i.e. an integral) is
estimated by random sampling.

### How fast does it converge?
Because each dart is an independent random trial, the statistical error of the estimate shrinks
only like $1/\sqrt{N}$ — the standard error is $\sqrt{\pi(4-\pi)}/\sqrt{N} \approx 1.64/\sqrt{N}$.
That is **slow**: to gain one more correct digit you need about **100× more** darts. (This is the
price of Monte Carlo — but the same $1/\sqrt{N}$ holds in *any* number of dimensions, which is why
Monte Carlo wins for high-dimensional integrals where grid methods become hopeless.)

### A deterministic yardstick: the BBP series
For contrast we also sum the **Bailey–Borwein–Plouffe (BBP)** series,

$$\pi = \sum_{k=0}^{\infty} 16^{-k}\left(\frac{4}{8k+1} - \frac{2}{8k+4} - \frac{1}{8k+5} - \frac{1}{8k+6}\right),$$

which converges to machine precision in about **13 terms** — a reminder of the gulf between a
*stochastic* estimate and a purpose-built *deterministic* formula.

## 1. Imports

The numerical core uses only the Python **standard library** (`math`, `random`). **matplotlib** is
the one third-party dependency (figures + animation, embedded inline via `jshtml`). The cell below
installs matplotlib if it is missing (handy on Google Colab), then imports everything.

In [ ]:
# --- Install required packages if missing (e.g. on Google Colab) ---
import importlib.util, subprocess, sys

for pkg in ["matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    else:
        print(f"{pkg} already available")

from math import acos, sqrt

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from matplotlib import rc

from random import random, seed

# Show animations inline (raise the embed-size limit so no frames get dropped)
rc('animation', html='jshtml', embed_limit=64)
%matplotlib inline

true_pi = acos(-1)   # reference value of pi
print("reference pi =", true_pi)

## 2. Parameters

`npoints` is the number of random darts. Larger `npoints` gives a better (but only slowly better)
estimate. The geometry matches the original program — a `BoxDim` square with the circle of radius
`Radius` at its centre — but the estimate itself does not depend on the size, only on the
*fraction* inside.

In [ ]:
npoints     = 100000       # number of random darts to throw
frame_every = 1000         # save one animation frame every this many darts (caps animation size/memory)
BoxDim  = [500.0, 500.0]  # square side (arbitrary units)
Center  = [250.0, 250.0]  # circle centre
Radius  = 250.0           # circle radius (inscribed in the square)
R2      = Radius**2        # radius squared (compare to squared distances)
Seed    = 100             # random-number seed (reproducibility)

## 3. Throw the darts

This headless loop replaces the GUI's `Go` callback / Tkinter event loop. Each step drops one
random point, tests whether it is inside the circle (using **squared** distances, so no `sqrt`),
updates the running estimate $\pi \approx 4 N_{\text{inside}}/N$ **every step**, and records the
history. Alongside it accumulates one term of the BBP series per step for comparison.

In [ ]:
def run_pi(npoints):
    seed(Seed)
    xs = []; ys = []; inside_flag = []      # dart positions and inside/outside
    est_hist = []; err_hist = []            # running MC estimate and its absolute error
    bbp_hist = []                           # running BBP series value
    n_inside = 0
    bbp = 0.0

    for n in range(1, npoints + 1):
        x = random()*BoxDim[0]
        y = random()*BoxDim[1]
        d2 = (x - Center[0])**2 + (y - Center[1])**2
        is_in = d2 <= R2
        if is_in:
            n_inside += 1

        est = 4.0 * n_inside / n            # Monte Carlo estimate of pi (updated every step)
        xs.append(x); ys.append(y); inside_flag.append(is_in)
        est_hist.append(est)
        err_hist.append(abs(true_pi - est))

        # add the k = n-1 term of the BBP series (first term is k = 0)
        k = n - 1
        bbp += 16.0**(-k) * (4.0/(8*k+1) - 2.0/(8*k+4) - 1.0/(8*k+5) - 1.0/(8*k+6))
        bbp_hist.append(bbp)

        if n in (10, 100, 1000, 10000, npoints):
            print("N = %7d   MC pi = %.5f   (error %.2e)   BBP pi = %.12f"
                  % (n, est, abs(true_pi - est), bbp))

    return xs, ys, inside_flag, est_hist, err_hist, bbp_hist


xs, ys, inside_flag, est_hist, err_hist, bbp_hist = run_pi(npoints)
n_in = sum(inside_flag)
print(f"\n{n_in} of {npoints} darts landed inside the circle.")
print(f"Final MC estimate: pi = {est_hist[-1]:.5f}  (true {true_pi:.5f})")

## 4. Convergence of the estimate

The left panel shows the running Monte Carlo estimate settling towards $\pi$ (dashed line) as darts
accumulate — noisily. The right panel plots the absolute error on **log–log** axes, where the
$1/\sqrt{N}$ Monte Carlo scaling shows up as a straight line of slope $-\tfrac12$ (grey reference).
The BBP series (orange) plunges to machine precision within ~13 terms — off the bottom of the plot —
dramatising how much faster a deterministic formula converges.

In [ ]:
Ns = range(1, npoints + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(Ns, est_hist, lw=1.0, color="tab:blue", label="MC estimate")
ax1.axhline(true_pi, ls="--", color="black", lw=1, label=r"true $\pi$")
ax1.set_xscale("log")
ax1.set_xlabel("number of darts N")
ax1.set_ylabel(r"estimate of $\pi$")
ax1.set_ylim(2.6, 3.7)
ax1.set_title(f"Running estimate (final {est_hist[-1]:.4f})")
ax1.legend()
ax1.grid(alpha=0.3)

# absolute error, log-log, with a 1/sqrt(N) reference and the BBP error
mc_err  = [max(e, 1e-6) for e in err_hist]
bbp_err = [max(abs(true_pi - b), 1e-16) for b in bbp_hist]
ref     = [1.64/sqrt(n) for n in Ns]
ax2.loglog(Ns, mc_err,  lw=1.0, color="tab:blue",   label="MC error")
ax2.loglog(Ns, ref,     ls="--", color="gray",       label=r"$1.64/\sqrt{N}$")
ax2.loglog(Ns, bbp_err, lw=1.2, color="tab:orange",  label="BBP error")
ax2.set_xlabel("number of terms / darts N")
ax2.set_ylabel(r"absolute error in $\pi$")
ax2.set_title("Error: Monte Carlo vs BBP series")
ax2.legend()
ax2.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

## 5. Visualise the darts

The darts, coloured by whether they landed **inside** (blue) or **outside** (red) the circle. The
blue points fill the disc; their share of the total is $\pi/4$.

In [ ]:
# draw a light, evenly-spaced subsample of the darts (keeps the figure legible and small)
disp_stride = max(1, npoints // 8000)
dx = xs[::disp_stride]; dy = ys[::disp_stride]; dh = inside_flag[::disp_stride]

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, BoxDim[0])
ax.set_ylim(0, BoxDim[1])
ax.set_aspect('equal')
ax.set_facecolor("#eef3ff")
ax.add_patch(Circle((Center[0], Center[1]), Radius, fill=False, edgecolor="black", lw=1.5))

xin  = [x for x, h in zip(dx, dh) if h]
yin  = [y for y, h in zip(dy, dh) if h]
xout = [x for x, h in zip(dx, dh) if not h]
yout = [y for y, h in zip(dy, dh) if not h]
n_in = sum(inside_flag)
ax.scatter(xin,  yin,  s=5, color="#3366cc", alpha=0.5, label="inside")
ax.scatter(xout, yout, s=5, color="#cc3333", alpha=0.5, label="outside")
ax.set_title(rf"{npoints} darts (showing {len(dx)})  →  $\pi\approx4\times${n_in}/{npoints} = {est_hist[-1]:.4f}")
ax.legend(loc="upper right", framealpha=0.9, markerscale=2)
plt.show()

## 6. Animation

Watch the darts accumulate and the estimate converge. This reproduces the live view of the original
program. To keep the animation small, **one frame is saved every `frame_every` = 1000 darts** (so
100 frames for 100 000 darts), and a light, evenly-spaced subsample of the points is drawn.

In [ ]:
frames = list(range(frame_every, npoints + 1, frame_every))   # one frame every `frame_every` darts

# animate a light, evenly-spaced subsample of the darts, revealed as N grows
disp_stride = max(1, npoints // 4000)
disp_pts  = list(zip(xs[::disp_stride], ys[::disp_stride]))
disp_cols = ["#3366cc" if h else "#cc3333" for h in inside_flag[::disp_stride]]

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, BoxDim[0])
ax.set_ylim(0, BoxDim[1])
ax.set_aspect('equal')
ax.set_facecolor("#eef3ff")
ax.add_patch(Circle((Center[0], Center[1]), Radius, fill=False, edgecolor="black", lw=1.5))
scat = ax.scatter([], [], s=5, alpha=0.5)
title = ax.set_title("")

def update(frame_idx):
    N = frames[frame_idx]
    ndisp = min(len(disp_pts), (N + disp_stride - 1)//disp_stride)
    scat.set_offsets(disp_pts[:ndisp])
    scat.set_facecolors(disp_cols[:ndisp])
    title.set_text(f"N = {N}   pi ≈ {est_hist[N-1]:.4f}")
    return scat, title

anim = FuncAnimation(fig, update, frames=len(frames), interval=80, blit=False)
plt.close(fig)   # avoid a duplicate static figure
anim

## 7. Monte Carlo vs a deterministic formula

The two methods here estimate the same number in completely different ways:

* **Monte Carlo (darts)** is dead simple and completely general — it just samples and counts — but
  its error falls only as $1/\sqrt{N}$. Reaching 4–5 correct digits takes *millions* of darts, and
  the estimate stays visibly noisy. That slowness is the trade-off for its generality.
* **The BBP series** is a purpose-built, deterministic formula that reaches machine precision in a
  handful of terms — but it works *only* for $\pi$; it teaches you nothing about sampling.

The point of the Monte Carlo notebooks in this course is the **first** idea: random sampling as a
way to compute averages and integrals. It converges slowly in this 2D toy example, but the very
same $1/\sqrt{N}$ scaling holds independently of dimension — which is exactly why Monte Carlo (and
the Metropolis algorithm in the companion `LJ-ELEC_MMC` notebooks) is the method of choice for the
huge-dimensional integrals of statistical mechanics.

Try raising `npoints`, or re-running with a different `Seed`, to see the estimate wander within its
$1/\sqrt{N}$ error band.